In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10fb66e40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10fb678c0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")

In [8]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10fb66e40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10fb678c0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the mo

In [7]:
model_with_structure.invoke("Please provide the details of the movie Inception.")

Movie(title='Inception', year=2010, director='Christopher Nolan')

In [9]:
model.invoke(
    "Please provide the title, year, and director of the movie 'Inception'.")

AIMessage(content='**Movie:** *Inception*  \n**Year:** 2010  \n**Director:** Christopher Nolan', additional_kwargs={'reasoning_content': 'The user asks for factual info about the movie "Inception". This is allowed. Provide title, year, director. So answer: Inception (2010), directed by Christopher Nolan. Provide title, year, director.'}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 87, 'total_tokens': 162, 'completion_time': 0.156966414, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.003219377, 'prompt_tokens_details': None, 'queue_time': 0.337118707, 'total_time': 0.160185791}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a075a3-6247-7c60-9f57-5630df1672e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 75, 'total_tokens': 162, 'output_token

Message output with parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    genre: str = Field(..., description="The genre of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke(
    "Please provide the title, year, director, and genre of the movie 'Inception'"
    )

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for title, year, director, genre of movie \'Inception\'. We have a function Movie that likely returns info. We need to call it with appropriate parameters: director Christopher Nolan, genre maybe Science Fiction/Action? The function expects genre string. We\'ll provide likely "Science Fiction". Let\'s call function.', 'tool_calls': [{'id': 'fc_0ee64dd6-ffe0-4ec2-a079-66ead7b833d7', 'function': {'arguments': '{"director":"Christopher Nolan","genre":"Science Fiction","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 167, 'total_tokens': 283, 'completion_time': 0.242499715, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.013727121, 'prompt_tokens_details': None, 'queue_time': 0.312754878, 'total_time': 0.256226836}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4

Nested Structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke(
    "Please provide details of the movie 'Inception'."
)
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], genre=['Science Fiction', 'Action', 'Thriller'], budget=160000000.0)

TypedDict

In [16]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie on a scale of 1 to 10"]

model_withtyped_dict = model.with_structured_output(Movie)
response = model_withtyped_dict.invoke(
    "Please provide the title, year, director, and rating of the movie Avengers"
)
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [17]:
model_with_basemode = model.with_structured_output(MovieDetails)
response = model_with_basemode.invoke(
    "Please provide details of the movie avengers."
)
response

MovieDetails(title='The Avengers', year=2012, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Mark Ruffalo', role='Bruce Banner / Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Jeremy Renner', role='Clint Barton / Hawkeye'), Actor(name='Tom Hiddleston', role='Loki'), Actor(name='Samuel L. Jackson', role='Nick Fury'), Actor(name='Clark Gregg', role='Phil Coulson'), Actor(name='Cobie Smulders', role='Maria Hill'), Actor(name='Stellan Skarsgård', role='Erik Selvig'), Actor(name='Gwyneth Paltrow', role='Pepper Potts'), Actor(name='Paul Bettany', role='JARVIS (voice)'), Actor(name='Robert Downey Sr.', role='Howard Stark (cameo)')], genre=['Action', 'Adventure', 'Sci-Fi'], budget=220000000.0)

In [18]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

DataClasses

In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(model="groq:openai/gpt-oss-120b", response_format=ContactInfo)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, johndoe@example.com, 123-456-7890"
    }]
})

result

{'messages': [HumanMessage(content='Extract contact information from: John Doe, johndoe@example.com, 123-456-7890', additional_kwargs={}, response_metadata={}, id='8176be87-0134-4ca6-8f0b-23d6beb75907'),
  AIMessage(content='{"name":"John Doe","email":"johndoe@example.com","phone":"123-456-7890"}', additional_kwargs={'reasoning_content': 'The user asks: "Extract contact information from: John Doe, johndoe@example.com, 123-456-7890". We need to output JSON matching the ContactInfo schema: fields name, email, phone required. Provide compact JSON. So output: {"name":"John Doe","email":"johndoe@example.com","phone":"123-456-7890"} Ensure it\'s compact, no extra whitespace.'}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 231, 'total_tokens': 354, 'completion_time': 0.26473763, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.009205225, 'prompt_tokens_details': None, 'queue_time': 0.341888352, 'total_time': 0.273942855}, 'model_name': '

In [25]:
from typing_extensions import Annotated, TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

agent = create_agent(model="groq:openai/gpt-oss-120b", response_format=ContactInfo)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, johndoe@example.com, 123-456-7890"
    }]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'johndoe@example.com', 'phone': '123-456-7890'}

In [26]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent(model="groq:openai/gpt-oss-120b", response_format=ContactInfo)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Extract contact information from: John Doe, johndoe@example.com, 123-456-7890"
    }]
})

result["structured_response"]

ContactInfo(name='John Doe', email='johndoe@example.com', phone='123-456-7890')